In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_spring_2026_svg_generation_from_text_prompts_extended_deadline_path = kagglehub.competition_download('dl-spring-2026-svg-generation-from-text-prompts-extended-deadline')
lavaghimire_qwen_svg_lora_v1_path = kagglehub.dataset_download('lavaghimire/qwen-svg-lora-v1')
lavaghimire_qwen_svg_merged_v1_path = kagglehub.dataset_download('lavaghimire/qwen-svg-merged-v1')
lavaghimire_qwen_svg_merged_v2_path = kagglehub.dataset_download('lavaghimire/qwen-svg-merged-v2')

print('Data source import complete.')


100%|██████████| 32.4M/32.4M [00:03<00:00, 8.58MB/s]

Extracting files...


100%|██████████| 90.0M/90.0M [00:08<00:00, 11.1MB/s]

Extracting files...


100%|██████████| 2.29G/2.29G [02:12<00:00, 18.6MB/s]

Extracting files...


100%|██████████| 2.29G/2.29G [03:20<00:00, 12.3MB/s]

Extracting files...


Data source import complete.


In [3]:
%pip install -q transformers peft accelerate bitsandbytes pandas tiktoken
%pip install sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.8 MB/s eta 0:00:00


In [4]:
# Imports
import re, time, xml.etree.ElementTree as ET
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"CUDA: {torch.cuda.is_available()}")
print(torch.cuda.get_device_name(0))

CUDA: True
NVIDIA A100-SXM4-40GB


In [5]:
# Load Data
#DATA_DIR = "/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline"
DATA_DIR = dl_spring_2026_svg_generation_from_text_prompts_extended_deadline_path

test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Test rows: {len(test_df)}")

Test rows: 1000


In [6]:
import os
print(os.listdir(f"{lavaghimire_qwen_svg_merged_v2_path}/qwen_svg_merged_v2"))

['tokenizer_config.json', 'config.json', 'model.safetensors', 'chat_template.jinja', 'tokenizer.json', '.cache']


In [ ]:
'''import os
import shutil

#MODEL_PATH = f"{lavaghimire_qwen_svg_merged_v1_path}/qwen_svg_merged"
MODEL_PATH = "/content/drive/MyDrive/svg_competition_v2/qwen_svg_merged_v2"


# Rename the weights file to the standard name
src = f"{MODEL_PATH}/model-001.safetensors"
dst = f"{MODEL_PATH}/model.safetensors"
shutil.copy(src, dst)
print("Renamed")
print(os.listdir(MODEL_PATH))'''

Renamed
['.cache', 'config.json', 'model-001.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors', 'chat_template.jinja']


In [7]:
#MODEL_PATH = "/kaggle/input/qwen-svg-merged-v1/qwen_svg_merged"
MODEL_PATH = f"{lavaghimire_qwen_svg_merged_v2_path}/qwen_svg_merged_v2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype      = torch.float16,
    device_map = "auto",
)
model.eval()
print("Model loaded")

The tokenizer you are loading from '/root/.cache/kagglehub/datasets/lavaghimire/qwen-svg-merged-v2/versions/1/qwen_svg_merged_v2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded


In [8]:
# Inference + Submission
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element, "
    "max 256x256 canvas, max 256 path elements."
)

SVG_RE = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)

def is_valid_svg(svg_text: str) -> bool:
    if not svg_text or not svg_text.strip().lower().startswith("<svg"):
        return False
    if len(svg_text) > 16000:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    return sum(1 for el in root.iter() if el.tag.split("}")[-1] == "path") <= 256

def extract_svg(text):
    m = SVG_RE.search(text)
    return m.group(0).strip() if m else ""

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="#fff"/>'
        '<circle cx="128" cy="128" r="80" fill="#333"/>'
        '</svg>'
    )

def generate_svg_batch(prompts, max_new_tokens=512):
    results = []
    for i, prompt in enumerate(prompts):
        chat = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = tokenizer(chat, return_tensors="pt", truncation=True,
                           max_length=512).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                use_cache      = True,   # works fine without unsloth
                repetition_penalty = 1.3,
                pad_token_id   = tokenizer.pad_token_id,
            )
        decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        svg = extract_svg(decoded)
        results.append(svg if is_valid_svg(svg) else fallback_svg())
        if i % 50 == 0:
            print(f"  {i+1}/{len(prompts)} done")
    return results

t0 = time.time()
svgs = generate_svg_batch(test_df["prompt"].tolist(), max_new_tokens=512)
sub_df = pd.DataFrame({"id": test_df["id"], "svg": svgs})
#sub_df.to_csv("submission.csv", index=False)  # save to current dir
sub_df.to_csv("/content/drive/MyDrive/submission.csv", index=False)
print("Saved to Drive ")
elapsed = (time.time() - t0) / 60
invalid = sum(1 for s in svgs if s == fallback_svg())
print(f"Done — {len(sub_df)} rows | fallbacks: {invalid} | {elapsed:.1f} min")
sub_df.head(3)

  1/1000 done
  51/1000 done
  101/1000 done
  151/1000 done
  201/1000 done
  251/1000 done
  301/1000 done
  351/1000 done
  401/1000 done
  451/1000 done
  501/1000 done
  551/1000 done
  601/1000 done
  651/1000 done
  701/1000 done
  751/1000 done
  801/1000 done
  851/1000 done
  901/1000 done
  951/1000 done


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive'

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
sub_df.to_csv("/content/drive/MyDrive/submission.csv", index=False)
print(f"Saved ✅ — {len(sub_df)} rows")

Saved ✅ — 1000 rows


In [11]:
# Validation
sub_path = "/content/drive/MyDrive/submission.csv"
assert os.path.exists(sub_path), "submission.csv not found!"
sub_df = pd.read_csv(sub_path)
assert len(sub_df) == 1000, f"Expected 1000 rows, got {len(sub_df)}"
assert list(sub_df["id"]) == list(test_df["id"]), "IDs don't match!"
valid = sub_df["svg"].apply(is_valid_svg).sum()
print(f"Rows: {len(sub_df)} | Valid SVGs: {valid} | Fallbacks: {1000-valid}")

Rows: 1000 | Valid SVGs: 1000 | Fallbacks: 0


In [12]:
# Check raw output for one prompt
prompt = test_df["prompt"].iloc[0]
chat = (
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\n{prompt}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)
inputs = tokenizer(chat, return_tensors="pt", truncation=True,
                   max_length=512).to(model.device)
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens = 512,
        do_sample      = False,
        use_cache      = True,
        pad_token_id   = tokenizer.pad_token_id,
    )
decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)
print(decoded)
print("---")
print(f"Output length: {len(decoded)} chars")
print(f"Contains </svg>: {'</svg>' in decoded.lower()}")

system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element, max 256x256 canvas, max 256 path elements.
user
firewood stack cut logs wood with leaf illustration.
assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" width="200" height="200" enable-background="new 0 0 200 200" xml:space="preserve"><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" fill-opacity="1"  d="M100 100h100v100h-100z"/><path fill="#292929" 